In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(
    "../prepared_data/reduced_vars_with_hmm.csv",
    index_col=0,
    parse_dates=True
)

In [3]:
TARGET_COL = "y_SP500_bin_4w"

y = df[TARGET_COL].copy()

X = df.drop(columns=[TARGET_COL], errors="ignore").copy()
X = X.drop(columns=["SP500"], errors="ignore")  # quita nivel del índice si existe

# Solo numéricas
X = X.select_dtypes(include=[np.number]).copy()

# Limpieza básica
X = X.replace([np.inf, -np.inf], np.nan)

data = X.join(y.rename("target")).dropna()
X = data.drop(columns=["target"])
y = data["target"]

print("X shape:", X.shape)
print("y value counts:\n", y.value_counts(dropna=False))

X shape: (1078, 36)
y value counts:
 target
1.0    857
0.0    221
Name: count, dtype: int64


In [4]:
# Make y numeric if binary as strings (IN/OUT)  -> PIPELINE: IN=0, OUT=1
if y.dtype == "object":
    y_mapped = y.map({"IN": 0, "OUT": 1})
    if y_mapped.isna().any():
        raise ValueError(f"Valores inesperados en y: {y.unique()}")
    y = y_mapped.astype(int)
else:
    # bool -> int, float-int -> int
    if y.dtype == "bool":
        y = y.astype(int)
    elif np.issubdtype(y.dtype, np.number):
        if np.all(np.isclose(y.values, y.values.astype(int))):
            y = y.astype(int)

print("y dtype:", y.dtype, "| unique:", np.unique(y))
print("Counts [IN=0, OUT=1]:", np.bincount(y))

# Optional: only if you KNOW y was already numeric but in the wrong convention (OUT=0, IN=1)
# y = 1 - y

y dtype: int64 | unique: [0 1]
Counts [IN=0, OUT=1]: [221 857]


In [5]:
CUTOFF_DATE = "2024-01-01"  # ajusta si tu compañero usa otra

X_train = X.loc[X.index < CUTOFF_DATE].copy()
X_test  = X.loc[X.index >= CUTOFF_DATE].copy()

y_train = y.loc[y.index < CUTOFF_DATE].copy()
y_test  = y.loc[y.index >= CUTOFF_DATE].copy()

print("Train:", X_train.index.min(), "->", X_train.index.max(), "| n =", len(X_train))
print("Test :", X_test.index.min(), "->", X_test.index.max(), "| n =", len(X_test))
print("y_train counts:\n", y_train.value_counts())
print("y_test counts:\n", y_test.value_counts())

Train: 2005-04-08 00:00:00 -> 2023-12-29 00:00:00 | n = 978
Test : 2024-01-05 00:00:00 -> 2025-11-28 00:00:00 | n = 100
y_train counts:
 target
1    771
0    207
Name: count, dtype: int64
y_test counts:
 target
1    86
0    14
Name: count, dtype: int64


Logistic Regression con Elastic Net

In [6]:
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score,
    confusion_matrix, classification_report, f1_score
)

In [7]:
tscv = TimeSeriesSplit(n_splits=3)

In [8]:
log_en = LogisticRegression(
    penalty="elasticnet",
    solver="saga",
    max_iter=5000,
    class_weight="balanced",   # clave para imbalance
    random_state=42
)

In [9]:
param_grid = {
    "C": [0.001, 0.01, 0.1, 1.0, 10.0],
    "l1_ratio": [0.05, 0.1, 0.3, 0.5, 0.7, 0.9]
}

grid_log = GridSearchCV(
    estimator=log_en,
    param_grid=param_grid,
    scoring="balanced_accuracy",
    cv=tscv,
    n_jobs=-1,
    verbose=1
)

grid_log.fit(X_train, y_train)

print("Best CV balanced_accuracy:", round(grid_log.best_score_, 4))
print("Best params:", grid_log.best_params_)

best_log = grid_log.best_estimator_

Fitting 3 folds for each of 30 candidates, totalling 90 fits


c:\Users\jmesc\NO_OneDrive\Portfolio_Opt\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Best CV balanced_accuracy: 0.6476
Best params: {'C': 0.01, 'l1_ratio': 0.1}


In [10]:
val_ratio = 0.2
split_val = int(len(X_train) * (1 - val_ratio))

X_tr, X_val = X_train.iloc[:split_val], X_train.iloc[split_val:]
y_tr, y_val = y_train.iloc[:split_val], y_train.iloc[split_val:]

best_log.fit(X_tr, y_tr)

val_proba = best_log.predict_proba(X_val)[:, 1]

thresholds = np.linspace(0.25, 0.75, 101)

best_thr = 0.5
best_f1_down = -1

for thr in thresholds:
    val_pred = (val_proba >= thr).astype(int)
    f1_down = f1_score(y_val, val_pred, pos_label=0)  # F1 para DOWN (0)

    if f1_down > best_f1_down:
        best_f1_down = f1_down
        best_thr = float(thr)

print("Best threshold:", round(best_thr, 3), "| Best F1 DOWN:", round(best_f1_down, 4))

c:\Users\jmesc\NO_OneDrive\Portfolio_Opt\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Best threshold: 0.46 | Best F1 DOWN: 0.396


In [11]:
# =====================================================
# STEP 6 — Test analytics (2024 onward) — BINARY (prettier confusion matrix)
# Model: Logistic Regression (Elastic Net)
# =====================================================

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    f1_score,
    accuracy_score,
    roc_auc_score
)

best_log.fit(X_train, y_train)

test_proba = best_log.predict_proba(X_test)[:, 1]   # p(OUT=1)
test_pred  = (test_proba >= best_thr).astype(int)

print("\n=== Logistic Regression (Elastic Net) (TEST) ===")
print("Accuracy:", round(accuracy_score(y_test, test_pred), 4))
print("Balanced Acc:", round(balanced_accuracy_score(y_test, test_pred), 4))
print("F1 (IN):", round(f1_score(y_test, test_pred, pos_label=1), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, test_proba), 4))

cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
cm_df = pd.DataFrame(
    cm,
    index=["True OUT (0)", "True IN (1)"],
    columns=["Pred OUT (0)", "Pred IN (1)"]
)

print("\nConfusion Matrix:")
print(cm_df)

print("\nClassification Report:")
print(classification_report(y_test, test_pred, target_names=["OUT (0)", "IN (1)"]))

c:\Users\jmesc\NO_OneDrive\Portfolio_Opt\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(



=== Logistic Regression (Elastic Net) (TEST) ===
Accuracy: 0.68
Balanced Acc: 0.7542
F1 (IN): 0.7778
ROC-AUC: 0.8389

Confusion Matrix:
              Pred OUT (0)  Pred IN (1)
True OUT (0)            12            2
True IN (1)             30           56

Classification Report:
              precision    recall  f1-score   support

     OUT (0)       0.29      0.86      0.43        14
      IN (1)       0.97      0.65      0.78        86

    accuracy                           0.68       100
   macro avg       0.63      0.75      0.60       100
weighted avg       0.87      0.68      0.73       100



#### Metrics

In [12]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

# --- TRAIN split (fit set)
p_tr = best_log.predict_proba(X_train)[:, 1]
pred_tr = (p_tr >= best_thr).astype(int)

# --- VAL split (threshold-tuning / validation set)
# If you already computed val_proba during threshold search, reuse it.
# Otherwise uncomment the next line.
# val_proba = best_log.predict_proba(X_val)[:, 1]
pred_va = (val_proba >= best_thr).astype(int)

# --- TEST (2024+)
p_te = test_proba  # already computed: best_log.predict_proba(X_test)[:, 1]
pred_te = test_pred  # already computed with best_thr

def r4(x):
    return float(f"{x:.4f}")

print(f"\n=== QUICK METRICS (0=IN, 1=OUT) @ thr={best_thr:.3f} ===")
print("TRAIN  acc:", r4(accuracy_score(y_train, pred_tr)),
      "bal_acc:", r4(balanced_accuracy_score(y_train, pred_tr)),
      "F1_OUT:", r4(f1_score(y_train, pred_tr, pos_label=1)))

print("VAL    acc:", r4(accuracy_score(y_val, pred_va)),
      "bal_acc:", r4(balanced_accuracy_score(y_val, pred_va)),
      "F1_OUT:", r4(f1_score(y_val, pred_va, pos_label=1)))

print("TEST   acc:", r4(accuracy_score(y_test, pred_te)),
      "bal_acc:", r4(balanced_accuracy_score(y_test, pred_te)),
      "F1_OUT:", r4(f1_score(y_test, pred_te, pos_label=1)))


=== QUICK METRICS (0=IN, 1=OUT) @ thr=0.460 ===
TRAIN  acc: 0.7147 bal_acc: 0.6724 F1_OUT: 0.8048
VAL    acc: 0.6888 bal_acc: 0.6227 F1_OUT: 0.7904
TEST   acc: 0.68 bal_acc: 0.7542 F1_OUT: 0.7778


In [13]:
# =========================
# Logistic Regression (Elastic Net) "feature importance"
# We use absolute coefficient magnitude as importance.
# NOTE: Coefs are only comparable if features are on similar scale
#       (usually you want StandardScaler in the pipeline).
# =========================

# If best_log is a Pipeline, try to grab the final estimator's coefficients
est = best_log
if hasattr(best_log, "named_steps"):  # sklearn Pipeline
    # grab last step (the classifier)
    est = list(best_log.named_steps.values())[-1]

coef = getattr(est, "coef_", None)

if coef is None:
    print("This model does not expose coef_. If you're using a pipeline, ensure the last step is LogisticRegression.")
else:
    # Binary logistic: shape (1, n_features) -> flatten
    coef = np.asarray(coef).ravel()

    imp_df = (
        pd.DataFrame({
            "feature": X_train.columns,
            "coef": coef,
            "abs_coef": np.abs(coef),
        })
        .sort_values("abs_coef", ascending=False)
        .reset_index(drop=True)
    )

    print("Total features with non-zero |coef|:", int((imp_df["abs_coef"] > 0).sum()))
    display(imp_df.head(40))

Total features with non-zero |coef|: 10


,feature,coef,abs_coef
0,NFCI__d1__z8,-0.352946,0.352946
1,NFCI__lvl__z8,-0.211215,0.211215
2,regime_label,-0.120332,0.120332
3,Natural_Gas__pct52,-0.091736,0.091736
4,WTI_Crude_Oil__pct52,-0.059508,0.059508
5,prob_state_0,0.051743,0.051743
6,resume_template__d26,-0.015489,0.015489
7,Expected_Inflation_5Y__lvl__L52,-0.007099,0.007099
8,sell_jewelry__lvl,-0.005349,0.005349
9,CPI_YoY__lvl__rollstd52,-0.003820,0.003820


### Data extraction

In [14]:
# =====================================================
# DF for trading sim (2024+ only) + simple checks + save
# Model: Logistic Regression (Elastic Net) (best_log)
# =====================================================

# Build df (2024+)
df_trading = df.loc[df.index >= CUTOFF_DATE].copy()

# Sanity: X_test index must match df_trading index (same dates, same order)
if not df_trading.index.equals(X_test.index):
    print("WARNING: df_trading.index != X_test.index")
    print("df_trading:", df_trading.index.min(), "->", df_trading.index.max(), "n=", len(df_trading))
    print("X_test    :", X_test.index.min(),     "->", X_test.index.max(),     "n=", len(X_test))
    missing_in_df = X_test.index.difference(df_trading.index)
    missing_in_X  = df_trading.index.difference(X_test.index)
    print("Missing in df_trading (should be 0):", len(missing_in_df))
    print("Missing in X_test (should be 0):", len(missing_in_X))
    if len(missing_in_df) > 0: print("Example missing_in_df:", missing_in_df[:5].tolist())
    if len(missing_in_X)  > 0: print("Example missing_in_X :", missing_in_X[:5].tolist())

# Add preds (aligned by index)
# NOTE:
# - test_proba: P(OUT=1) from best_log.predict_proba(X_test)[:, 1]
# - test_pred : (test_proba >= best_thr).astype(int)
df_trading["p_out"] = pd.Series(test_proba, index=X_test.index)
df_trading["pred_out"] = pd.Series(test_pred, index=X_test.index)
df_trading["y_out_true"] = pd.Series(y_test, index=X_test.index)

# NaN checks (just the important columns)
nan_counts = df_trading[["p_out", "pred_out", "y_out_true"]].isna().sum()
print("\nNaNs in key cols:\n", nan_counts)

# quick assertion-like prints
print("\nRows in df_trading:", len(df_trading))
print("Pred rows (X_test):", len(X_test))
print("All key cols non-null? ->", (nan_counts.sum() == 0))

# Save
out_path = "../predictions/logreg_preds.csv"
df_trading.to_csv(out_path)
print("\nSaved ->", out_path)


NaNs in key cols:
 p_out         0
pred_out      0
y_out_true    0
dtype: int64

Rows in df_trading: 100
Pred rows (X_test): 100
All key cols non-null? -> True

Saved -> ../predictions/logreg_preds.csv
